# YOLOv8 License Plate Training Notebook

This notebook trains `yolov8n` to detect license plates from images.

Update the values in the config cell once, then run the notebook top to bottom.

Assumptions:
- Your dataset is in YOLO format.
- Your dataset has a `data.yaml` file.
- The task is single-class detection: `license_plate`.


## Dataset Structure Exploration

Before configuring paths, let's inspect how the dataset is organized. This will help us decide which folder to use for training.

In [ ]:
from pathlib import Path
import shutil
import torch
from ultralytics import YOLO

In [ ]:
CONFIG = {
    "project_root": Path(r"D:\PATH\TO\YOUR\PROJECT"),
    "dataset_yaml": Path(r"D:\PATH\TO\YOUR\DATASET\data.yaml"),
    "model_name": "yolov8n.pt",
    "run_name": "license_plate_yolov8n",
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "patience": 20,
    "workers": 4,
    "seed": 42,
    "confidence": 0.25,
    "iou": 0.7,
    "sample_image": Path(r"D:\PATH\TO\A\SAMPLE\IMAGE.jpg"),
}

CONFIG["project_root"] = CONFIG["project_root"].resolve()
CONFIG["dataset_yaml"] = CONFIG["dataset_yaml"].resolve()
CONFIG["sample_image"] = CONFIG["sample_image"].resolve()
CONFIG["runs_dir"] = CONFIG["project_root"] / "runs"
CONFIG["train_dir"] = CONFIG["project_root"] / "train_outputs"

for key in ("project_root", "dataset_yaml", "sample_image"):
    print(f"{key}: {CONFIG[key]}")


In [ ]:
# Basic validation before training.
assert CONFIG["project_root"].exists(), f"Missing project_root: {CONFIG['project_root']}"
assert CONFIG["dataset_yaml"].exists(), f"Missing dataset_yaml: {CONFIG['dataset_yaml']}"
assert CONFIG["sample_image"].exists(), f"Missing sample_image: {CONFIG['sample_image']}"

CONFIG["project_root"], CONFIG["dataset_yaml"], CONFIG["sample_image"]


In [ ]:
# Train YOLOv8n on the license plate dataset.
# The YAML file should point to train/val image folders and names for the class list.
model = YOLO(CONFIG["model_name"])

train_results = model.train(
    data=str(CONFIG["dataset_yaml"]),
    epochs=CONFIG["epochs"],
    imgsz=CONFIG["imgsz"],
    batch=CONFIG["batch"],
    device=CONFIG["device"],
    patience=CONFIG["patience"],
    workers=CONFIG["workers"],
    seed=CONFIG["seed"],
    project=str(CONFIG["runs_dir"]),
    name=CONFIG["run_name"],
    exist_ok=True,
)

train_results

In [ ]:
# Save the trained model in two forms:
# 1) YOLO checkpoint (.pt) for easy reuse with YOLO(...)
# 2) state_dict (.pt) for PyTorch-based loading
save_dir = CONFIG["project_root"] / "saved_models"
save_dir.mkdir(parents=True, exist_ok=True)

run_save_dir = Path(train_results.save_dir)
best_weights = run_save_dir / "weights" / "best.pt"

saved_files = []

if best_weights.exists():
    model_target = save_dir / f"{CONFIG['run_name']}_model.pt"
    shutil.copy2(best_weights, model_target)
    saved_files.append(model_target)
    print(f"Saved YOLO checkpoint to: {model_target}")

    state_dict_target = save_dir / f"{CONFIG['run_name']}_state_dict.pt"
    torch.save(model.model.state_dict(), state_dict_target)
    saved_files.append(state_dict_target)
    print(f"Saved state_dict to: {state_dict_target}")
else:
    raise FileNotFoundError(f"Missing trained checkpoint: {best_weights}")

print(f"Saved {len(saved_files)} file(s) in: {save_dir}")
saved_files

In [ ]:
# Validate the trained model on the validation split.
metrics = model.val(data=str(CONFIG["dataset_yaml"]), imgsz=CONFIG["imgsz"], device=CONFIG["device"])
metrics


In [ ]:
# Run inference on one sample image.
# After training, YOLO keeps the best checkpoint under the run directory.
best_weights = CONFIG["runs_dir"] / CONFIG["run_name"] / "weights" / "best.pt"
assert best_weights.exists(), f"Missing trained weights: {best_weights}"

trained_model = YOLO(str(best_weights))
predictions = trained_model.predict(
    source=str(CONFIG["sample_image"]),
    conf=CONFIG["confidence"],
    iou=CONFIG["iou"],
    save=True,
    project=str(CONFIG["project_root"] / "predictions"),
    name=CONFIG["run_name"],
    exist_ok=True,
)

predictions


## Notes

- If your dataset uses a different class name, update `data.yaml` and keep the notebook config unchanged.
- If you are training on CPU, set `device` to `"cpu"`.
- If you want to try a bigger model later, change `model_name` to `yolov8s.pt` or another YOLOv8 checkpoint.
- For a single-class license plate detector, your label files should contain bounding boxes for plate regions only.
